# 📊 Task 4: Evaluate LLM Outputs — BLEU, ROUGE & Perplexity

### What this notebook does (step by step):
1. Installs all required libraries
2. Takes **5 prompts** and sends them to **GPT-4o, Gemini, and Claude** via real API calls
3. Computes **BLEU**, **ROUGE-1/2/L**, and **Perplexity** scores on the responses
4. Displays a **comparison chart** and **summary table**
5. Downloads a **metric score report CSV**

---
### What each metric means:
| Metric | Measures | Good score |
|--------|----------|------------|
| **BLEU** | N-gram overlap with reference answer | Higher is better (0–1) |
| **ROUGE-1** | Word overlap with reference | Higher is better (0–1) |
| **ROUGE-2** | 2-word phrase overlap | Higher is better (0–1) |
| **ROUGE-L** | Longest matching sequence | Higher is better (0–1) |
| **Perplexity** | How fluent/natural the text sounds | Lower is better |

## ⚙️ Step 1: Install Libraries
Run this once. It installs everything needed.

In [ ]:
!pip install openai google-generativeai anthropic nltk rouge-score transformers torch pandas matplotlib -q

import nltk
nltk.download('punkt',     quiet=True)
nltk.download('punkt_tab', quiet=True)

print('✅ All libraries installed!')

## 🔑 Step 2: Enter Your API Keys

You need free API keys from:
- **OpenAI (GPT-4o):** https://platform.openai.com/api-keys
- **Google (Gemini):** https://aistudio.google.com/app/apikey
- **Anthropic (Claude):** https://console.anthropic.com/settings/keys

> All three have free tiers. Just sign up and copy your key.

In [ ]:
# ── PASTE YOUR KEYS HERE ──────────────────────────────────────────────
OPENAI_API_KEY    = "sk-..."       # from platform.openai.com
GOOGLE_API_KEY    = "AIza..."      # from aistudio.google.com
ANTHROPIC_API_KEY = "sk-ant-..."   # from console.anthropic.com
# ──────────────────────────────────────────────────────────────────────

import openai
import google.generativeai as genai
import anthropic

openai_client  = openai.OpenAI(api_key=OPENAI_API_KEY)
genai.configure(api_key=GOOGLE_API_KEY)
claude_client  = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)

print('✅ API clients ready!')

## 📝 Step 3: Define Prompts + Reference Answers

We use 5 prompts across different categories.
Each prompt has a **reference answer** — this is the "correct" answer we compare model outputs against when computing BLEU and ROUGE.

In [ ]:
PROMPTS = [
    {
        "id"       : "P1",
        "category" : "Factual Knowledge",
        "prompt"   : "Explain what a transformer neural network is in 3 sentences.",
        "reference": (
            "A transformer is a neural network architecture that uses self-attention "
            "mechanisms to process sequential data. It was introduced in the 2017 paper "
            "Attention Is All You Need. Transformers are the foundation of modern large "
            "language models like GPT, BERT, and Claude."
        )
    },
    {
        "id"       : "P2",
        "category" : "Reasoning",
        "prompt"   : "A train travels 120 km in 1.5 hours. What is its average speed? Show your working.",
        "reference": (
            "Speed equals distance divided by time. "
            "120 divided by 1.5 equals 80. "
            "The average speed of the train is 80 km/h."
        )
    },
    {
        "id"       : "P3",
        "category" : "Creative Writing",
        "prompt"   : "Write a 2-sentence opening to a science fiction story set on Mars.",
        "reference": (
            "The rust-red horizon stretched endlessly under a pale sun that gave light but little warmth. "
            "Commander Reyes pressed her gloved hand against the dome and wondered if "
            "anyone on Earth still remembered their names."
        )
    },
    {
        "id"       : "P4",
        "category" : "Code Generation",
        "prompt"   : "Write a Python function that takes a list of numbers and returns the second largest value.",
        "reference": (
            "def second_largest(nums): "
            "    unique = sorted(set(nums), reverse=True); "
            "    return unique[1] if len(unique) >= 2 else None"
        )
    },
    {
        "id"       : "P5",
        "category" : "Ethical Reasoning",
        "prompt"   : "Should AI be used to make hiring decisions? Give a brief pros and cons answer.",
        "reference": (
            "Pros: AI can process more candidates consistently and reduce individual human bias. "
            "Cons: AI can encode historical biases, lacks judgment for culture fit, "
            "and raises serious fairness and legal concerns."
        )
    },
]

print(f'✅ {len(PROMPTS)} prompts defined.')
for p in PROMPTS:
    print(f"  {p['id']} — {p['category']}")

## 🚀 Step 4: Query All Three Models

This cell sends each prompt to GPT-4o, Gemini 1.5 Flash, and Claude 3.5 Haiku and collects their responses.

> This may take 1–2 minutes to complete.

In [ ]:
import time

# ── Model query functions ─────────────────────────────────────────────

def query_gpt(prompt):
    """GPT-4o via OpenAI API"""
    try:
        resp = openai_client.chat.completions.create(
            model="gpt-4o",
            messages=[{"role": "user", "content": prompt}],
            max_tokens=300,
            temperature=0.7
        )
        return resp.choices[0].message.content.strip()
    except Exception as e:
        return f"[GPT Error: {e}]"

def query_gemini(prompt):
    """Gemini 1.5 Flash via Google API"""
    try:
        model = genai.GenerativeModel('gemini-1.5-flash')
        resp  = model.generate_content(prompt)
        return resp.text.strip()
    except Exception as e:
        return f"[Gemini Error: {e}]"

def query_claude(prompt):
    """Claude 3.5 Haiku via Anthropic API"""
    try:
        msg = claude_client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=300,
            messages=[{"role": "user", "content": prompt}]
        )
        return msg.content[0].text.strip()
    except Exception as e:
        return f"[Claude Error: {e}]"

# ── Run all prompts ───────────────────────────────────────────────────
import pandas as pd

records = []
for item in PROMPTS:
    print(f"\n🔄 {item['id']} — {item['category']}")

    gpt_out    = query_gpt(item['prompt'])
    gemini_out = query_gemini(item['prompt'])
    claude_out = query_claude(item['prompt'])

    print(f"   GPT-4o    → {gpt_out[:90]}...")
    print(f"   Gemini    → {gemini_out[:90]}...")
    print(f"   Claude    → {claude_out[:90]}...")

    records.append({
        'Prompt ID' : item['id'],
        'Category'  : item['category'],
        'Prompt'    : item['prompt'],
        'Reference' : item['reference'],
        'GPT-4o'    : gpt_out,
        'Gemini 1.5': gemini_out,
        'Claude 3.5': claude_out,
    })
    time.sleep(1)   # small delay to avoid rate limits

df = pd.DataFrame(records)
print(f'\n✅ All {len(df)} prompts completed!')

## 🔵 Step 5: Compute BLEU Scores

**BLEU** counts how many words and phrases (n-grams) from the model's answer match the reference answer.
- Score closer to **1.0** = very similar to reference
- Score closer to **0.0** = very different from reference

In [ ]:
from nltk.translate.bleu_score import sentence_bleu, SmoothingFunction
from nltk.tokenize import word_tokenize

smoother = SmoothingFunction().method1

def bleu(reference, hypothesis):
    ref_tokens = word_tokenize(reference.lower())
    hyp_tokens = word_tokenize(hypothesis.lower())
    return round(
        sentence_bleu([ref_tokens], hyp_tokens,
                      weights=(0.25, 0.25, 0.25, 0.25),
                      smoothing_function=smoother), 4
    )

df['BLEU_GPT']    = df.apply(lambda r: bleu(r['Reference'], r['GPT-4o']),    axis=1)
df['BLEU_Gemini'] = df.apply(lambda r: bleu(r['Reference'], r['Gemini 1.5']),axis=1)
df['BLEU_Claude'] = df.apply(lambda r: bleu(r['Reference'], r['Claude 3.5']),axis=1)

print('✅ BLEU scores computed:')
print(df[['Category','BLEU_GPT','BLEU_Gemini','BLEU_Claude']].to_string(index=False))

## 🟢 Step 6: Compute ROUGE Scores

**ROUGE** measures how much of the reference answer is covered by the model's response:
- **ROUGE-1**: individual word overlap
- **ROUGE-2**: two-word phrase overlap
- **ROUGE-L**: longest matching word sequence

In [ ]:
from rouge_score import rouge_scorer

rscorer = rouge_scorer.RougeScorer(['rouge1','rouge2','rougeL'], use_stemmer=True)

def rouge(reference, hypothesis):
    s = rscorer.score(reference, hypothesis)
    return {
        'r1': round(s['rouge1'].fmeasure, 4),
        'r2': round(s['rouge2'].fmeasure, 4),
        'rl': round(s['rougeL'].fmeasure, 4),
    }

for model, tag in [('GPT-4o','GPT'), ('Gemini 1.5','Gemini'), ('Claude 3.5','Claude')]:
    scores = df.apply(lambda r: rouge(r['Reference'], r[model]), axis=1)
    df[f'R1_{tag}'] = scores.apply(lambda x: x['r1'])
    df[f'R2_{tag}'] = scores.apply(lambda x: x['r2'])
    df[f'RL_{tag}'] = scores.apply(lambda x: x['rl'])

print('✅ ROUGE scores computed:')
print(df[['Category','R1_GPT','R1_Gemini','R1_Claude']].to_string(index=False))

## 🟠 Step 7: Compute Perplexity

**Perplexity** measures how natural and fluent the model's response sounds.
We use a small GPT-2 model as the judge.
- **Lower score** = text sounds more natural and fluent
- **Higher score** = text sounds unusual or unnatural

> Note: Creative writing will always score higher perplexity — that's expected and normal.

In [ ]:
import torch
from transformers import GPT2LMHeadModel, GPT2TokenizerFast

print('Loading GPT-2 (used only to measure fluency — takes ~30 seconds)...')
ppl_tokenizer = GPT2TokenizerFast.from_pretrained('gpt2')
ppl_model     = GPT2LMHeadModel.from_pretrained('gpt2')
ppl_model.eval()
print('✅ GPT-2 loaded!')

def perplexity(text):
    encodings = ppl_tokenizer(
        text, return_tensors='pt', truncation=True, max_length=512
    )
    input_ids = encodings.input_ids
    with torch.no_grad():
        loss = ppl_model(input_ids, labels=input_ids).loss
    return round(torch.exp(loss).item(), 2)

print('\nComputing perplexity scores (this takes ~1 minute)...')
for model, tag in [('GPT-4o','GPT'), ('Gemini 1.5','Gemini'), ('Claude 3.5','Claude')]:
    df[f'PPL_{tag}'] = df[model].apply(perplexity)
    print(f'  ✅ {model} done')

print('\n✅ Perplexity scores computed:')
print(df[['Category','PPL_GPT','PPL_Gemini','PPL_Claude']].to_string(index=False))

## 📊 Step 8: Visualize All Metrics

In [ ]:
import matplotlib.pyplot as plt
import numpy as np

cats   = df['Category'].tolist()
x      = np.arange(len(cats))
w      = 0.25
colors = {'GPT': '#10A37F', 'Gemini': '#4285F4', 'Claude': '#D96A00'}

fig, axes = plt.subplots(2, 2, figsize=(16, 11))
fig.suptitle('LLM Output Evaluation: BLEU, ROUGE & Perplexity',
             fontsize=16, fontweight='bold', y=0.99)

plots = [
    (axes[0,0], 'BLEU Score  (Higher = Better)',
     [('GPT','BLEU_GPT'),('Gemini','BLEU_Gemini'),('Claude','BLEU_Claude')], True),
    (axes[0,1], 'ROUGE-1 F1  (Higher = Better)',
     [('GPT','R1_GPT'),('Gemini','R1_Gemini'),('Claude','R1_Claude')], True),
    (axes[1,0], 'ROUGE-L F1  (Higher = Better)',
     [('GPT','RL_GPT'),('Gemini','RL_Gemini'),('Claude','RL_Claude')], True),
    (axes[1,1], 'Perplexity  (Lower = More Fluent)',
     [('GPT','PPL_GPT'),('Gemini','PPL_Gemini'),('Claude','PPL_Claude')], False),
]

for ax, title, series, has_ylim in plots:
    for i, (tag, col) in enumerate(series):
        bars = ax.bar(x + i*w, df[col], w, label=tag,
                      color=colors[tag], alpha=0.85)
        for bar in bars:
            ax.text(bar.get_x() + bar.get_width()/2,
                    bar.get_height() + 0.005,
                    f'{bar.get_height():.3f}',
                    ha='center', va='bottom', fontsize=7.5)
    ax.set_title(title, fontweight='bold')
    ax.set_xticks(x + w)
    ax.set_xticklabels(cats, rotation=18, ha='right', fontsize=9)
    ax.legend()
    ax.grid(axis='y', alpha=0.3)
    if has_ylim:
        ax.set_ylim(0, 1)

plt.tight_layout()
plt.savefig('metrics_chart.png', dpi=150, bbox_inches='tight')
plt.show()
print('✅ Chart saved as metrics_chart.png')

## 📋 Step 9: Final Summary Table

In [ ]:
from IPython.display import display, HTML

summary = pd.DataFrame({
    'Model'         : ['GPT-4o', 'Gemini 1.5 Flash', 'Claude 3.5 Haiku'],
    'Avg BLEU'      : [df['BLEU_GPT'].mean(), df['BLEU_Gemini'].mean(), df['BLEU_Claude'].mean()],
    'Avg ROUGE-1'   : [df['R1_GPT'].mean(),   df['R1_Gemini'].mean(),   df['R1_Claude'].mean()],
    'Avg ROUGE-2'   : [df['R2_GPT'].mean(),   df['R2_Gemini'].mean(),   df['R2_Claude'].mean()],
    'Avg ROUGE-L'   : [df['RL_GPT'].mean(),   df['RL_Gemini'].mean(),   df['RL_Claude'].mean()],
    'Avg Perplexity': [df['PPL_GPT'].mean(),  df['PPL_Gemini'].mean(),  df['PPL_Claude'].mean()],
}).round(4)

# Highlight best score in each column
def highlight_best(col):
    if col.name == 'Avg Perplexity':
        best = col.min()
    elif col.name == 'Model':
        return ['' for _ in col]
    else:
        best = col.max()
    return ['background-color:#C6EFCE; font-weight:bold' if v == best else '' for v in col]

styled = summary.style.apply(highlight_best)
display(styled)

# Print plain version too
print('\n📊 Summary (green = best in column):')
print(summary.to_string(index=False))

## 💾 Step 10: Download Your Reports

In [ ]:
# Save detailed report (all prompts + all scores)
detail_cols = [
    'Prompt ID','Category',
    'BLEU_GPT','BLEU_Gemini','BLEU_Claude',
    'R1_GPT','R1_Gemini','R1_Claude',
    'R2_GPT','R2_Gemini','R2_Claude',
    'RL_GPT','RL_Gemini','RL_Claude',
    'PPL_GPT','PPL_Gemini','PPL_Claude'
]
df[detail_cols].to_csv('metric_score_report.csv', index=False)
summary.to_csv('metric_summary.csv', index=False)

print('✅ Files saved!')

from google.colab import files
files.download('metric_score_report.csv')   # detailed scores per prompt
files.download('metric_summary.csv')        # average scores per model
files.download('metrics_chart.png')         # the bar chart

print('📥 Downloads started — check your browser downloads folder!')